# Project 12 GraphGPS Upgrade Synthetic Validation

This private kernel runs only provider-free synthetic checks. It uses no datasets, internet, GPU, model weights, checkpoints, W&B, or uploads. It stops at explicit approval gates before any external or heavy action.

In [ ]:
import hashlib
import importlib.metadata as metadata
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
ROOT = Path.cwd()
search_roots = [ROOT, Path('/kaggle/working'), Path('/kaggle/input')]
search_roots.extend(ROOT.parents)
source_candidates = []
for base in search_roots:
    for candidate in [base, base / 'source', *base.glob('*'), *base.glob('*/source')]:
        if candidate.is_dir() and (candidate / 'graphgps_bench').is_dir():
            source_candidates.append(candidate)
if not source_candidates:
    visible = [str(path) for base in search_roots[:3] if base.exists() for path in list(base.iterdir())[:20]]
    raise RuntimeError({'source_only_package_missing': True, 'visible_runtime_entries': visible})
SOURCE_TREE = sorted(set(source_candidates), key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(SOURCE_TREE))
print({'python': sys.version, 'platform': platform.platform(), 'cwd': str(ROOT), 'source_tree': str(SOURCE_TREE)})

In [ ]:
restricted_suffixes = {'.pt', '.pth', '.safetensors', '.parquet', '.jsonl', '.ckpt'}
restricted_parts = {'data', 'datasets', 'checkpoints', 'wandb'}
restricted_names = {'.env', 'credentials.json', 'kaggle.json'}
restricted = []
for path in SOURCE_TREE.rglob('*'):
    if not path.is_file():
        continue
    relative = path.relative_to(SOURCE_TREE)
    if path.suffix.lower() in restricted_suffixes or path.name in restricted_names or any(part in restricted_parts for part in relative.parts):
        restricted.append(str(relative).replace('\\', '/'))
if restricted:
    raise RuntimeError({'restricted_artifacts': restricted})
print({'source_file_count': sum(1 for path in SOURCE_TREE.rglob('*') if path.is_file()), 'restricted_artifact_count': 0})

In [ ]:
from graphgps_bench.config import load_config

config_path = SOURCE_TREE / 'configs' / 'fixture.toml'
config = load_config(config_path)
config_hash = config.stable_hash()
source_digest = hashlib.sha256()
for path in sorted(SOURCE_TREE.rglob('*')):
    if path.is_file():
        source_digest.update(str(path.relative_to(SOURCE_TREE)).encode())
        source_digest.update(path.read_bytes())
source_revision = source_digest.hexdigest()
def package_version(module_name, distribution_name=None):
    if importlib.util.find_spec(module_name) is None:
        return None
    try:
        return metadata.version(distribution_name or module_name)
    except metadata.PackageNotFoundError:
        return 'importable-without-metadata'
torch_version = package_version('torch')
pyg_version = package_version('torch_geometric', 'torch-geometric')
ogb_version = package_version('ogb')
cuda_available = False
if torch_version:
    import torch
    cuda_available = bool(torch.cuda.is_available())
device = 'cpu'
print({'config_hash': config_hash, 'source_revision': source_revision, 'torch': torch_version, 'torch_geometric': pyg_version, 'ogb': ogb_version, 'cuda_available': cuda_available, 'device': device, 'seed': config.seeds[0]})

In [ ]:
def run(command):
    completed = subprocess.run(command, cwd=SOURCE_TREE, env=os.environ.copy(), text=True, capture_output=True)
    return {'command': command, 'exit_code': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr}

compile_result = run([sys.executable, '-m', 'compileall', '-q', 'graphgps_bench', 'tests'])
test_result = run([sys.executable, '-m', 'pytest', '-p', 'no:cacheprovider', '-q', 'tests'])
test_text = test_result['stdout'] + test_result['stderr']
test_match = re.search(r'(\d+) passed', test_text)
test_count = int(test_match.group(1)) if test_match else 0
if compile_result['exit_code'] != 0 or test_result['exit_code'] != 0:
    raise RuntimeError({'compile_exit_code': compile_result['exit_code'], 'test_exit_code': test_result['exit_code']})
print({'compile_exit_code': compile_result['exit_code'], 'test_exit_code': test_result['exit_code'], 'test_count': test_count})

In [ ]:
import torch
from graphgps_bench.data import make_fixture_batch
from graphgps_bench.models import ModelSpec, build_model

fixture = make_fixture_batch(seed=config.seeds[0]).to_torch()
model = build_model(ModelSpec(name='gps', input_dim=fixture['x'].size(-1), edge_dim=fixture['edge_attr'].size(-1), hidden_dim=config.gps.hidden_dim, gps_recipe=config.gps)).eval()
with torch.no_grad():
    output = model(fixture['x'], fixture['edge_index'], fixture['edge_attr'], fixture['batch'])
if tuple(output.shape) != (fixture['y'].size(0), 1) or not bool(torch.isfinite(output).all()):
    raise RuntimeError({'output_shape': tuple(output.shape), 'finite': bool(torch.isfinite(output).all())})
print({'synthetic_graph_count': fixture['num_graphs'] if hasattr(fixture, 'num_graphs') else int(fixture['y'].size(0)), 'gps_output_shape': tuple(output.shape), 'gps_smoke_exit_code': 0})

In [ ]:
evidence_path = ROOT / 'graphgps_upgrade_evidence.json'
evidence = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'source_revision': source_revision,
    'configuration_hash': config_hash,
    'dependency_versions': {'python': platform.python_version(), 'torch': torch_version, 'torch_geometric': pyg_version, 'ogb': ogb_version},
    'device': device,
    'cuda_available': cuda_available,
    'seed': config.seeds[0],
    'synthetic_sample_count': int(fixture['y'].size(0)),
    'test_count': test_count,
    'exit_code': 0,
    'compile_exit_code': compile_result['exit_code'],
    'test_exit_code': test_result['exit_code'],
    'gps_smoke_exit_code': 0,
    'artifact_paths': [str(evidence_path)],
    'restricted_artifact_count': 0,
    'benchmark_executed': False,
    'gpu_used': False,
}
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(json.loads(evidence_path.read_text(encoding='utf-8')), indent=2, sort_keys=True))

## Approval gate

The synthetic evidence above is the stopping point for this kernel. The next cells remain blocked until a matching approval is recorded.

In [ ]:
raise RuntimeError('Approval required before dependency installation or verification')

In [ ]:
# APPROVAL REQUIRED BEFORE RUNNING.
raise RuntimeError('Approval required before dependency installation or verification')
import inspect
import torch_geometric
from torch_geometric.nn import GPSConv
print({'torch_geometric': torch_geometric.__version__, 'GPSConv': inspect.signature(GPSConv)})

In [ ]:
raise RuntimeError('Approval required before OGB download/access')

In [ ]:
raise RuntimeError('Approval required before GPU execution/training')